# Analyze Stage-2 Cases

Aggregates all 12 case outputs (`H00`..`H11`), prints summary table, ranking, and Tier-A eval commands.

In [2]:
from __future__ import annotations

import csv
import json
import shlex
from pathlib import Path

_cwd = Path.cwd().resolve()
if (_cwd / 'src' / 'train.py').is_file():
    PROJECT_ROOT = _cwd
elif (_cwd / 'case_runner.py').is_file():
    PROJECT_ROOT = _cwd.parents[1]
else:
    PROJECT_ROOT = _cwd

DATA_ROOT = PROJECT_ROOT / 'data'
TOP_K = 5

RUNS = [
    {'name': 'H00_baseline', 'overrides': {}},
    {'name': 'H01_lr_low', 'overrides': {'lr': '1e-4'}},
    {'name': 'H02_lr_high', 'overrides': {'lr': '3e-4'}},
    {'name': 'H03_batch_16', 'overrides': {'batch-size': '16'}},
    {'name': 'H04_epochs_plus', 'overrides': {'epochs': '55'}},
    {'name': 'H05_drop_sketch_high', 'overrides': {'drop-sketch-prob': '0.2'}},
    {'name': 'H06_guidance_high', 'overrides': {'guidance-scale': '2.0'}},
    {'name': 'H07_sample_steps_fast', 'overrides': {'sample-steps': '120'}},
    {'name': 'H08_lpips_late', 'overrides': {'lpips-start-frac': '0.25'}},
    {'name': 'H09_color_strong', 'overrides': {'color-loss-weight': '0.05', 'color-loss-start-frac': '0.45'}},
    {'name': 'H10_linear_schedule', 'overrides': {'beta-schedule': 'linear'}},
    {'name': 'H11_min_snr_5', 'overrides': {'min-snr-gamma': '5'}},
]

def _last_val_row(metrics_csv: Path) -> dict[str, str]:
    if not metrics_csv.is_file():
        return {}
    rows = list(csv.DictReader(metrics_csv.open(newline='')))
    for r in reversed(rows):
        if (r.get('val_loss') or '').strip():
            return r
    return {}

def _safe_float(x: str) -> float | None:
    try:
        return float(x)
    except (TypeError, ValueError):
        return None

def summarize_checkpoint_dir(name: str, save_dir: Path) -> dict:
    mpath = save_dir / 'metrics.csv'
    row = _last_val_row(mpath)
    eval_path = save_dir / 'test_eval_summary.json'
    eval_blob: dict = {}
    if eval_path.is_file():
        try:
            eval_blob = json.loads(eval_path.read_text())
        except json.JSONDecodeError:
            eval_blob = {'_error': 'invalid json'}
    out = {
        'run': name,
        'save_dir': str(save_dir),
        'val_loss': _safe_float(row.get('val_loss', '')),
        'val_psnr': _safe_float(row.get('val_psnr', '')),
        'last_step': int(row['step']) if row.get('step', '').isdigit() else None,
        'eval_keys': list(eval_blob.keys()) if eval_blob else [],
    }
    out['test_psnr_db'] = eval_blob.get('test_psnr_db') if eval_blob else None
    out['test_ddpm_loss'] = eval_blob.get('test_ddpm_loss') if eval_blob else None
    out['test_fid'] = eval_blob.get('fid') if eval_blob else None
    return out

summary_rows: list[dict] = []
for run in RUNS:
    sd = PROJECT_ROOT / 'checkpoints' / f"hp_{run['name']}"
    summary_rows.append(summarize_checkpoint_dir(run['name'], sd))

hdr = f"{'run':<22} {'val_loss':>10} {'val_psnr':>10} {'test_psnr':>10} {'test_fid':>10} {'step':>8} eval"
print(hdr)
print('-' * len(hdr))
for s in summary_rows:
    vl = f"{s['val_loss']:.6f}" if s['val_loss'] is not None else 'n/a'
    vp = f"{s['val_psnr']:.2f}" if s['val_psnr'] is not None else 'n/a'
    tp = f"{s['test_psnr_db']:.2f}" if s.get('test_psnr_db') is not None else 'n/a'
    tf = s.get('test_fid')
    try:
        tfs = f"{float(tf):.2f}" if tf is not None else 'n/a'
    except (TypeError, ValueError):
        tfs = 'n/a'
    st = str(s['last_step'] if s['last_step'] is not None else 'n/a')
    ej = 'yes' if s['eval_keys'] else 'no'
    print(f"{s['run']:<22} {vl:>10} {vp:>10} {tp:>10} {tfs:>10} {st:>8} {ej:>8}")

def sort_key(s: dict):
    vl = s['val_loss']
    vp = s['val_psnr']
    tp = s.get('test_psnr_db')
    return (
        float('inf') if vl is None else vl,
        float('-inf') if vp is None else -vp,
        float('-inf') if tp is None else -tp,
    )

ranked = sorted(summary_rows, key=sort_key)
registry = {r['name']: r for r in RUNS}
print(f'Top {TOP_K} by val_loss (tie-break val_psnr up, test_psnr up):')
for i, s in enumerate(ranked[:TOP_K], 1):
    ov = registry.get(s['run'], {}).get('overrides', {})
    extras = ' '.join(f'{k}={v}' for k, v in sorted(ov.items())) if ov else '(baseline bundle)'
    vl = f"{s['val_loss']:.6f}" if s['val_loss'] is not None else 'n/a'
    vp = f"{s['val_psnr']:.2f}" if s['val_psnr'] is not None else 'n/a'
    print(f"  {i}. {s['run']}: val_loss={vl} val_psnr={vp} | {extras}")

print('\nTier A eval grid commands for top candidates:')
for s in ranked[:TOP_K]:
    ckpt = Path(s['save_dir']) / 'ckpt_best.pt'
    cmd = ' '.join(
        [
            f"CKPT={shlex.quote(str(ckpt))}",
            f"GENAI_ROOT={shlex.quote(str(PROJECT_ROOT))}",
            f"EVAL_ROOT={shlex.quote(str(PROJECT_ROOT))}",
            f"DATA_ROOT={shlex.quote(str(DATA_ROOT))}",
            f"GRID_SUFFIX={shlex.quote('hp_' + s['run'])}",
            'bash scripts/run_tier_a_eval_grid.sh',
        ]
    )
    print(cmd)


run                      val_loss   val_psnr  test_psnr   test_fid     step eval
--------------------------------------------------------------------------------
H00_baseline             0.017219      28.55      29.12      88.47   168399      yes
H01_lr_low               0.017991      28.49      29.04      95.41   168399      yes
H02_lr_high              0.017893      28.56      29.12     100.52   168399      yes
H03_batch_16             0.018118      28.59      29.13      93.31   336875      yes
H04_epochs_plus          0.018161      28.29      28.29     184.21   120285      yes
H05_drop_sketch_high     0.017297      28.38      28.94     107.77   168399      yes
H06_guidance_high        0.017458      28.38      28.94     180.67   168399      yes
H07_sample_steps_fast    0.017408      28.52      28.95      72.75   168399      yes
H08_lpips_late           0.017387      28.48      28.99      97.93   168399      yes
H09_color_strong         0.018681      28.23      28.14     127.04   1683